# Recurrent Neural Network

In [ ]:
import sys
import os

# Agregar el directorio raiz al PYTHONPATH
module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
from src.trainers.utils import build_datasets
from constants.constants_twitter import TWITTER_DATASET_TRAIN_PATH

dataset_train, dataset_test, dataset_val = build_datasets(
    TWITTER_DATASET_TRAIN_PATH,
    test_size=0.3,
    val_size=0.5, # 0.5 de 0.3    
    random_state=42,
    undersampling=True
)
print(dataset_train.shape)
print(dataset_test.shape)
print(dataset_val.shape)

In [ ]:
from src.preprocesamiento.nlp_spacy import Tokenizer

tokenizer = Tokenizer("en")

dataset_train_tokenized = {}
dataset_val_tokenized = {}
dataset_test_tokenized = {}

dataset_train_tokenized['tokens'] = tokenizer.tokenize(dataset_train['text'], True)
dataset_val_tokenized['tokens'] = tokenizer.tokenize(dataset_val['text'], True)
dataset_test_tokenized['tokens'] = tokenizer.tokenize(dataset_test['text'], True)

dataset_train_tokenized['polarity'] = dataset_train['polarity'].to_numpy()
dataset_val_tokenized['polarity'] = dataset_val['polarity'].to_numpy()
dataset_test_tokenized['polarity'] = dataset_test['polarity'].to_numpy()

## Entrenar RNN

In [ ]:
import numpy as np
from sklearn.model_selection import ParameterGrid
from src.trainers.utils import EarlyStopping

param_grid = {
  "optim": ["adam", "sgd"],
  "lr": np.logspace(-3, -0.3, 10)
}

combinaciones = list(ParameterGrid(param_grid))
for idx, comb in enumerate(combinaciones, 1):
    print(f"Model {idx}: {comb}")

epochs = 200
patience = 15
min_delta = 1e-4
hp = {}
hp['batch_size'] = 64

model_args = {}
model_args['hidden_size'] = 128
model_args['num_layers'] = 2
model_args['output_size'] = 3
model_args['dropout'] = 0.20

In [ ]:
from src.trainers.train_rnn import train_rnn
from src.trainers.utils import save_metrics, show_loss_val_curves, save_model_torch
from constants.constants_twitter import RNN_LOSS_CURVES_DIR, RNN_MODEL_DIR, RNN_METRICS_PATH, EMBEDDING_W2V_TWITTER_PATH

best_accuracy = -1
for idx, param in enumerate(combinaciones, 1):
    model, metrics, train_losses, val_losses = train_rnn(
        dataset_train=dataset_train_tokenized,
        dataset_val=dataset_val_tokenized,
        embeddings_path=EMBEDDING_W2V_TWITTER_PATH,
        model_args=model_args,
        early_stopping = EarlyStopping(patience, min_delta), # reinicio
        batch_size=hp['batch_size'],
        lr=param['lr'],
        epochs=epochs,
        optim=param['optim'],
    )

    metrics['name'] = f"model_{idx}"
    print(f"[{metrics['name']} Acc: {metrics['accuracy']:.4f}] {param}")
    save_metrics(metrics, RNN_METRICS_PATH)

    model_args['input_size'] = metrics['embedding_dim']
    save_model_torch(model.get_model(), model_args, hp, os.path.join(RNN_MODEL_DIR, f"{metrics['name']}.pt"))
  
    title = f"Pérdida de entrenamiento y validación"
    path = os.path.join(RNN_LOSS_CURVES_DIR, f"{metrics['name']}.png")
    show_loss_val_curves(train_losses, val_losses, len(train_losses), title, path)

    if metrics['accuracy'] > best_accuracy:
        best_accuracy = metrics['accuracy']
        print(f"New best accuracy ({metrics['name']})\n")
        save_model_torch(model.get_model(), model_args, hp, os.path.join(RNN_MODEL_DIR, "best_model.pt"))

## Modelo con mayor accuracy

In [ ]:
import pandas as pd

# Seleccionar los hiperparámetros que generan mayor accuracy
df_metrics = pd.read_csv(RNN_METRICS_PATH)

best_acc = df_metrics.loc[df_metrics['accuracy'].idxmax()]
print(best_acc)

In [ ]:
import torch
from src.trainers.train_rnn import RNNModel
from src.trainers.trainer_rnn import evaluate_model

checkpoint = torch.load(os.path.join(RNN_MODEL_DIR, "best_model.pt"))
config = checkpoint['config']
hp = checkpoint['hp']

model = RNNModel(**config)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval() # Inferencia (desactiva dropout)

metrics = evaluate_model(
    model,
    dataset_test_tokenized,
    "RNN",
    EMBEDDING_W2V_TWITTER_PATH,
    hp['batch_size']
)
display(metrics)